In [0]:
# =============================================================================
# 1. PARÁMETROS DE ENTRADA (NO MODIFICAR)
# Valores inyectados por el orquestador (Databricks Jobs / Bundle)
# =============================================================================
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json
import sys

# --- Widgets de Databricks para ejecución orquestada ---
dbutils.widgets.text("periodo", "202512", "Periodo")
dbutils.widgets.text("mode_environment", "dev", "Mode Environment")

periodo = dbutils.widgets.get("periodo")
mode_environment = dbutils.widgets.get("mode_environment")

env = mode_environment
mes_vta = int(periodo) # <=== MES DATA = MES CERRADO BCP. Ejemplo, para el mes venta=202605, entonces mes data debe ser 202603

print(f"Ejecutando notebook para el periodo: {mes_vta} en entorno: {env}")

In [0]:
# =============================================================================
# 2. CONFIGURACIÓN EDITABLE - MLE
# Variables principales que deben ser adaptadas para cada nuevo modelo o caso de uso.
# =============================================================================

# --- 2.1 Workspace y MLFlow ---
WORKSPACE = "/Workspace/Users/williammunoz@bcp.com.pe/cemm-dtbr-deploy-bhv-troncal-cliente/src"
# WORKSPACE = "/Workspace/Users/mariarojasgarzon@bcp.com.pe/cemm-dtbr-deploy-bhv-troncal-cliente/src" # _INPUT

# --- 2.2 Metadatos del Modelo y variables utilizadas ---
USE_CASE = "BHV_TRONCAL_CLIENTE"           # Nombre o identificador del caso de uso
PREFIX = "UM_BHV_GAHI_VTA"                 # Prefijo identificador estándar de variables

CAMPO_PRIMARIO = "codclavepartycli"        # Llave primaria del cliente
CAMPOS = ["xb_bhv_idio_gahi", "xb_bhv_gahi_3q24"] # Variables base utilizadas

SCORE_NAME = "SC_BHV_GAHI_3Q24"            # Nombre final a presentar del score
SCORE_NAME_MODEL = "numscore"              # Nombre de la columna de score output del predict

TABLE_MODEL = "gbs_test_arnold"            # Str. Nombre exacto de la tabla de modelo generada (como array)

# --- 2.3 Nombres Base de Tablas ---
# Nota: La base de datos/esquema y el entorno se agregarán automáticamente en la sección 3
BASE_NAME_TABLE_MASTER = "bhv_troncal_cliente_base"
BASE_NAME_TABLE_OUTPUT = "bcp_scoring_bhv_troncal_cliente"

# --- 2.4 Comportamiento de Ejecución y Debugging ---
RUN_HISTORICAL = False                    # True para ejecutar el backtest histórico en batch. OBS: se sobreescribe a FIRST_RUN
FIRST_RUN = True                          # Flag de primer run para cálculos de calidad
MESES_HISTORICOS_CALIDAD = 2              # Rango en meses para validación histórica de datos
DEBUG_MLOPS = False                       # Activar límite de muestras (Debug logs)
DEBUG_ROWS = 1000                         # Cantidad límite de filas procesadas si Debug es True
NUM_ERRORS = 0                            # Control de errores inicial

# --- 2.5 Configuración de Segmentación Múltiple (Opcional) ---
USE_SEGMENTS = False                      # Habilitar si la predicción rutea a distintos modelos por regla
SEGMENTS_CONFIG = []                      # no aplica

# --- 2.6 Certificación ---
CRITICAL_FIELDS = []                      # Definir campos críticos si aplica
JOIN_KEYS = ['codclavepartycli']
EXCLUDE_COLUMNS = []
CERTIFICATION_THRESHOLDS = {
    'records': {'green': 99.0},
    'schema': {'green': 100.0},
    'data_match': {'green': 99.0, 'yellow': 96.0},
    'critical_fields': {'green': True}
}

# Construcción de la tabla base de comparación. Adhoc para este proyecto
from pyspark.sql import functions as F
config_tabla_scoring = "catalog_lhcl_prod.bcp.bcp_edv_fabseg.bhv_port_d_mod_dev_gb_out_v2_history"
config_tabla_seguimiento = "catalog_lhcl_prod.bcp.bcp_edv_fabseg.T36436x_bhv_troncal_hm_portafolio_rd_base_hist_1012"
config_codmes_list = [202510, 202511, 202512]
config_cols_final = [
    'codmes',
    'codclavepartycli',
    'codinternocomputacional',
    'codclaveunicocli',
    'flgclictavalida',
    'ctddiaatraso',
    'max_maduracion_cli',
    'mtosaldocapitalsol',
    'FLG_TC',
    'FLG_TC_PERSONAS',
    'FLG_CEF',
    'FLG_VEH',
    'FLG_HIP',
    'ctdmora_intra_0_imp_cut15',
    'max_maduracion_cli_imp150',
    'max_mora_intra_u6m',
    'exp_pct_evol_ship_u3m_rt_u6m_cua',
    'prd_pct_pmpas_pmact_24_24_rt24a',
    'fatc_pct_pag_mn_ctamin_u6m_rtu6',
    'rcc_deu_ind_pj_prm_u3m_impt',
    'rcc_pct_sf3_sf24_ship_rt_u24',
    'rcc_pct_rdv_prm_u3ma',
    'prd_prm_tsav_mnn_6_6_rt6a',
    'mto_deu_mora_sol_u48_log',
    'flg_titulo_c',
    'q_diamora_max_100_u24a',
    'ftc_flg_pg_ful_clant_sol_mx_u3_c',
    'rcc_pct_sf12_sf24_rt_u24c',
    'edad_cut',
    'rcc_q_mes_act_sf_buen_mal_0_u3_3',
    'evol_mora_intra_g3m_cut',
    'ctdpdhu24_cut',
    'pos_pct_q_etcnpscl_a_sum_u6_rt6',
    'pos_tkt_trx_com_sol_prm_u3ma',
    'grf_tip_clas_rie_cli_2_4_mx_u3m',
    'isav_q_opea_desm_prm_u3m_cut',
    'evol_deu_ship_max_u12m_u24_raw',
    'grf_cvta_prov_rie4_prm_u3m',
    'prod_flg_sld_aho_300',
    'q_mes_mto_tot_pgsrv_sol_m0_u3m',
    'rcc_mto_gar_ope_cre_cut_logr',
    'numpd',
    'numxb',
    'numsc'
]

config_df_scg = spark.table(config_tabla_scoring).filter(F.col("codmes").isin(config_codmes_list))
config_df_scg = config_df_scg.withColumn("numxb", F.log(F.col("numpd") / (1 - F.col("numpd"))))
config_df_scg = config_df_scg.withColumn(
    "numsc",
    F.when(
        F.col("numxb").isNotNull(),
        F.greatest( F.lit(3), F.round(174.25 - 57.708 * F.col("numxb"), 1) )
    )
)

config_df_seg = spark.table(config_tabla_seguimiento).filter(F.col("codmes").isin(config_codmes_list))
config_df_scg_seg = config_df_scg.join(config_df_seg, ['codmes', 'codclavepartycli'], 'inner')
config_df_scg_seg = config_df_scg_seg.select(*config_cols_final)

SCORING_TABLE_SAS = config_df_scg_seg

In [0]:
# =============================================================================
# 3. CONFIGURACIÓN COMPLETA DEL SISTEMA MLOPS (NO MODIFICAR RUTINARIAMENTE)
# Resolución de Storage, schemas y catalogos basados en el environment
# =============================================================================

ENVIRONMENT_CONFIG = {
    "dev": {
        "CONTAINER": "bcp-expl-007",
        "STORAGE_ACCOUNT": "adlscuilemmbackp07",
        "CATALOG_NAME": "catalog_cemm_expl_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_007",
        "SCHEMA_MODELS": "bcp_expl_007_models",
    },
    "stg": {
        "CONTAINER": "bcp-expl-007",
        "STORAGE_ACCOUNT": "adlscuilemmbackp07",
        "CATALOG_NAME": "catalog_cemm_expl_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_007",
        "SCHEMA_MODELS": "bcp_expl_007_models",
    },
    "prod": {
        "CONTAINER": "bcp-expl-008",
        "STORAGE_ACCOUNT": "adlscuilemmbackp08",
        "CATALOG_NAME": "catalog_cemm_desp_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_008",
        "SCHEMA_MODELS": "bcp_expl_008_models",
    },
}

if env not in ENVIRONMENT_CONFIG:
    raise ValueError(f"Ambiente '{env}' no válido. Use 'dev', 'stg' o 'prod'")

_env_cfg = ENVIRONMENT_CONFIG[env]
CONTAINER = _env_cfg["CONTAINER"]
STORAGE_ACCOUNT = _env_cfg["STORAGE_ACCOUNT"]
CATALOG_NAME = _env_cfg["CATALOG_NAME"]
SCHEMA_NAME = _env_cfg["SCHEMA_NAME"]
SCHEMA_MODELS = _env_cfg["SCHEMA_MODELS"]

print(f"Configuración de ambiente '{env}' cargada: Catalog={CATALOG_NAME}, Schema={SCHEMA_NAME}")

# Construcción de nombres finales de tablas orquestadas
BASE_NAME_TABLE_MASTER_ENV = f"{BASE_NAME_TABLE_MASTER}_{env}"
TABLE_MASTER = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER_ENV}"
BASE_NAME_TABLE_OUTPUT_ENV = f"{BASE_NAME_TABLE_OUTPUT}_{env}"
TABLE_OUTPUT = BASE_NAME_TABLE_OUTPUT_ENV

In [0]:
# CONFIGURACIÓN AUTOMÁTICA - MLE No modificar

# --- Variables del Modelo ---
use_case = USE_CASE
prefix = PREFIX
campo_primario = CAMPO_PRIMARIO
campos = CAMPOS
campos_task = json.dumps(CAMPOS)
score_name = SCORE_NAME
score_name_model = SCORE_NAME_MODEL
num_errors = NUM_ERRORS
DEBUG_MLOPS = DEBUG_MLOPS
debug_rows = DEBUG_ROWS
first_run = FIRST_RUN
num_meses_historicos = MESES_HISTORICOS_CALIDAD

# --- Variables de Segmentos (derivadas) ---
use_segments = USE_SEGMENTS
segments_config = SEGMENTS_CONFIG
num_segments = len(SEGMENTS_CONFIG) if USE_SEGMENTS else 0

# --- Configuración de Storage (derivada) ---
container = CONTAINER
storage = STORAGE_ACCOUNT
work_path = f"{env}/bcp/ddv/mlops/mmgr"
adls_location_table = f"abfss://{container}@{storage}.dfs.core.windows.net/{work_path}"

# --- Configuración de Catálogo (derivada) ---
catalog_name = CATALOG_NAME
schema_name = SCHEMA_NAME
table_name_mt = TABLE_MASTER

# --- Nombres Completos de Tablas ---
model_table = f"{catalog_name}.{SCHEMA_MODELS}.{TABLE_MODEL}"
output_table = f"{catalog_name}.{schema_name}.{TABLE_OUTPUT}"

# --- Project Path ---
try:
    project_path = '/Workspace' + os.path.dirname(
        os.path.dirname(
            dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        )
    )
except Exception:
    project_path = WORKSPACE

sys.path.insert(0, str(project_path))
from utils.month_delta import month_delta

# --- Cálculo de Meses Relativos ---
mes_vta_1 = month_delta(str(mes_vta), -1)
mes_vta_2 = month_delta(str(mes_vta), -2)
mes_vta_3 = month_delta(str(mes_vta), -3)

# --- Paths ADLS Base ---
tmp_path = adls_location_table
tmp_path_mt = f"{adls_location_table}/tmp/{mes_vta}_mt"

# --- Paths ADLS Data Out ---
data_out_path = f"{adls_location_table}/data/out/{mes_vta}"
data_out_path_1 = f"{adls_location_table}/data/out/{mes_vta_1}"
data_out_path_2 = f"{adls_location_table}/data/out/{mes_vta_2}"
data_out_path_3 = f"{adls_location_table}/data/out/{mes_vta_3}"

# --- Paths ADLS Reports ---
reports_path_input = f"{adls_location_table}/archive/reports/input/{mes_vta}"
reports_path_output = f"{adls_location_table}/archive/reports/output/{mes_vta}"

# --- Paths Locales (Workspace) ---
data_path = f"{project_path}/data"
logs_path = f"{project_path}/logs"
utils_path = f"{project_path}/utils"
quality_config = f"{project_path}/config/quality_config"

# --- Paths de Datos Locales ---
template_report_path = f"{project_path}/data/reports/template"
mapping_path = f"{project_path}/data/mapping"
json_path = f"{project_path}/data/json"
parquet_path = f"{project_path}/data/parquet"

# --- Paths Temporales Locales ---
temp_reports_output_path = f"{project_path}/data/reports/tmp/output/{mes_vta}"
temp_reports_input_path = f"{project_path}/data/reports/tmp/input/{mes_vta}"
temp_reports_certification_path = f"{project_path}/data/reports/tmp/certification/{mes_vta}"

# --- Paths de Pipelines ---
data_processing_pipeline = f"{project_path}/pipelines/01_preprocessing_pipeline.py"
model_inference_pipeline = f"{project_path}/pipelines/02_inference_pipeline.py"
model_result_pipeline = f"{project_path}/pipelines/03_result_pipeline.py"

meses_historicos_calidad = MESES_HISTORICOS_CALIDAD

print("=" * 90)
print("✓ Configuración cargada exitosamente")
print("=" * 90)